# Scraping Google search results: AI Copyright and Governance

## 0. Install Dependencies

In [1]:
!pip install google-search-results pandas python-dotenv requests trafilatura youtube_transcript_api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 6.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [trafilatura] [trafilatura]ean]


## 1. Imports and Configurations

In [9]:
from serpapi import GoogleSearch
from youtube_transcript_api import YouTubeTranscriptApi
from concurrent.futures import ThreadPoolExecutor, as_completed
import trafilatura
import pandas as pd
from dotenv import load_dotenv
import os
import re
import time

load_dotenv()
SERP_API_KEY = os.getenv("SERPAPI_KEY")

# Max number of articles to scrape
MAX_NUM_RESULTS = 10

## SerpAPI - Collecting article and video URLs

In [10]:
def collect_search_results(queries: list[str],
                           num_results: int = MAX_NUM_RESULTS) -> tuple[pd.DataFrame, list[dict]]:
    """
    Search Google for each query and collect:
    - organic_results  → articles (title, snippet, url, source, date)
    - inline_videos    → YouTube videos (title, channel, url)

    Returns:
        articles_df  : DataFrame of all article metadata
        raw_results  : list of raw SerpAPI result dicts (just for inspection)
    """
    article_rows = []
    video_rows   = []
    raw_results  = []

    for query in queries:
        print(f"\n🔍 Searching: '{query}'")

        params = {
            "engine":        "google",
            "q":             query,
            "google_domain": "google.com",
            "hl":            "en",
            "gl":            "us",
            "api_key":       SERP_API_KEY
        }

        search  = GoogleSearch(params)
        results = search.get_dict()
        raw_results.append({"query": query, "results": results})

        # ── Organic articles ──
        organic = results.get("organic_results", [])
        if not organic:
            print(f"  ⚠️  No organic_results for: {query}")
        else:
            for article in organic[:num_results]:
                article_rows.append({
                    "query":   query,
                    "title":   article.get("title"),
                    "snippet": article.get("snippet"),
                    "source":  article.get("source"),
                    "date":    article.get("date"),
                    "url":     article.get("link"),
                    "type":    "article"
                })
            print(f"  ✅ {len(organic[:num_results])} articles collected")

        # ── Inline videos ──
        videos = results.get("inline_videos", [])
        if videos:
            for video in videos:
                video_rows.append({
                    "query":   query,
                    "title":   video.get("title"),
                    "snippet": None,
                    "source":  video.get("channel"),
                    "date":    None,
                    "url":     video.get("link"),
                    "type":    "video"
                })
            print(f"  ✅ {len(videos)} videos collected")

    # Combine into single DataFrame
    all_rows = article_rows + video_rows
    df = pd.DataFrame(all_rows)

    if df.empty:
        print("\n⚠️  No results collected across all queries.")
        return df, raw_results

    # Remove duplicates by URL
    df = df.drop_duplicates(subset="url").reset_index(drop=True)

    return df, raw_results

In [12]:
# ── Collect urls from search query ──
df, raw = collect_search_results(["AI Intellectual Property", 
                                  "Copyright Generative AI"])
df.head()


🔍 Searching: 'AI Intellectual Property'
  ✅ 8 articles collected

🔍 Searching: 'Copyright Generative AI'
  ✅ 9 articles collected


,query,title,snippet,source,date,url,type
0,AI Intellectual Property,"AI, Copyright, and the Law: The Ongoing Battle...",Artificial intelligence (AI) is rapidly reshap...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article
1,AI Intellectual Property,Artificial Intelligence and Intellectual Property,AI inventions present the current patent syste...,World Intellectual Property Organization (WIPO),NaN,https://www.wipo.int/en/web/frontier-technolog...,article
2,AI Intellectual Property,Generative AI: Navigating intellectual property,Artificial intelligence challenges the traditi...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article
3,AI Intellectual Property,What's yours isn't mine: AI and intellectual p...,If AI uses their intellectual property without...,JHU Carey Business School,"Jun 14, 2024",https://carey.jhu.edu/news/whats-yours-isnt-mi...,article
4,AI Intellectual Property,Generative Artificial Intelligence and Copyrig...,The AI Guidance states that authors may claim ...,Congress.gov,"Jul 18, 2025",https://www.congress.gov/crs-product/LSB10922,article


## 3. Scraping text from articles and videos

In [13]:
# Define scrapers
def _scrape_url(url: str) -> dict:
    """Scrape clean article text from a URL using Trafilatura."""
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return {"url": url, "full_text": None,
                    "status": "failed_download"}

        text = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=True,
            no_fallback=False,
            favor_precision=False,
            deduplicate=True
        )
        return {
            "url":       url,
            "full_text": text.strip() if text else None,
            "status":    "success" if text else "failed_extraction"
        }
    except Exception as e:
        return {"url": url, "full_text": None,
                "status": f"error: {str(e)}"}


def _get_youtube_id(url: str) -> str | None:
    """Extract YouTube video ID from URL."""
    patterns = [
        r"youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})",
        r"youtube\.com/shorts/([a-zA-Z0-9_-]{11})",
        r"youtu\.be/([a-zA-Z0-9_-]{11})"
    ]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    return None


def _get_transcript(video_id: str) -> str | None:
    """Fetch YouTube transcript from video ID."""
    try:
        transcript = YouTubeTranscriptApi.get_transcript(
            video_id,
            languages=["en", "en-US", "en-GB"]
        )
        return " ".join([t["text"] for t in transcript])
    except Exception:
        return None


# ═══════════════════════════════════════════════════════
# ENRICHMENT
# ═══════════════════════════════════════════════════════

def enrich_search_results(df: pd.DataFrame,
                         max_workers: int = 5,
                         delay: float = 1.0) -> pd.DataFrame:
    """
    Enrich a DataFrame from collect_search_results() with full text content.
    - Rows with type='article' → scraped via Trafilatura
    - Rows with type='video'   → transcript via YouTube Transcript API

    Args:
        df:          DataFrame returned by collect_articles()
        max_workers: concurrent threads for article scraping
        delay:       seconds between requests per thread

    Returns:
        Enriched DataFrame with added columns:
        - full_text  : scraped article text OR video transcript
        - video_id   : YouTube video ID (videos only)
        - status     : success / failed_download / no_transcript / etc.
    """
    if df.empty:
        print("⚠️  Empty DataFrame — nothing to enrich.")
        return df

    df = df.copy()
    df["full_text"] = None
    df["video_id"]  = None
    df["status"]    = "pending"

    # ── Split by type ──
    article_mask = df["type"] == "article"
    video_mask   = df["type"] == "video"

    article_df = df[article_mask].copy()
    video_df   = df[video_mask].copy()

    # ── Scrape articles ──
    if not article_df.empty:
        print(f"\n📰 Scraping {len(article_df)} articles...")

        url_to_result = {}

        def fetch_with_delay(url):
            time.sleep(delay)
            return _scrape_url(url)

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(fetch_with_delay, row["url"]): idx
                for idx, row in article_df.iterrows()
            }
            for i, future in enumerate(as_completed(futures)):
                result = future.result()
                url_to_result[result["url"]] = result
                icon = "✅" if result["status"] == "success" else "❌"
                source = df.loc[futures[future], "source"] or "Unknown"
                print(f"  {icon} [{i+1}/{len(article_df)}] "
                      f"{source} — {result['status']}")

        # Map results back
        for idx, row in article_df.iterrows():
            r = url_to_result.get(row["url"], {})
            df.at[idx, "full_text"] = r.get("full_text")
            df.at[idx, "status"]    = r.get("status", "unknown")

        success = sum(
            1 for r in url_to_result.values()
            if r["status"] == "success"
        )
        print(f"\n  📊 Articles: {success}/{len(article_df)} "
              f"scraped successfully")

    # ── Fetch video transcripts ──
    if not video_df.empty:
        print(f"\n🎬 Fetching transcripts for "
              f"{len(video_df)} videos...")

        for idx, row in video_df.iterrows():
            url      = row["url"]
            vid_id   = _get_youtube_id(url)
            transcript = None

            if vid_id:
                transcript = _get_transcript(vid_id)

            status = "success" if transcript else "no_transcript"
            icon   = "✅" if transcript else "❌"

            df.at[idx, "video_id"]  = vid_id
            df.at[idx, "full_text"] = transcript
            df.at[idx, "status"]    = status

            title = (row["title"] or "")[:50]
            print(f"  {icon} {title}... [{status}]")

        success = sum(
            1 for idx in video_df.index
            if df.at[idx, "status"] == "success"
        )
        print(f"\n  📊 Videos: {success}/{len(video_df)} "
              f"transcripts fetched")

    return df

In [14]:
# ── Enrich data with full texts from articles and videos ──
df = enrich_search_results(df)

# ── Filter to only successfully scraped content ──
df_clean = df[df["status"] == "success"].reset_index(drop=True)

# ── Inspect results ──
df_clean.head()


📰 Scraping 15 articles...
  ❌ [1/15] Congress.gov — failed_download
  ❌ [2/15] JHU Carey Business School — failed_download
  ✅ [3/15] Nixon Peabody — success
  ✅ [4/15] World Intellectual Property Organization (WIPO) — success
  ✅ [5/15] University of Southern California — success
  ❌ [6/15] American Bar Association — failed_download
  ❌ [7/15] Reddit · r/law — failed_extraction
  ✅ [8/15] Copyright Office (.gov) — success
  ✅ [9/15] Medium · Adnan Masood, PhD. — success
  ✅ [10/15] National Institutes of Health (NIH) | (.gov) — success
  ✅ [11/15] The Regulatory Review — success
  ✅ [12/15] University of South Florida — success
  ❌ [13/15] ScienceDirect.com — failed_download
  ✅ [14/15] The University of Virginia — success
  ✅ [15/15] Harvard Magazine — success

  📊 Articles: 10/15 scraped successfully


,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI Intellectual Property,"AI, Copyright, and the Law: The Ongoing Battle...",Artificial intelligence (AI) is rapidly reshap...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,None,success
1,AI Intellectual Property,Artificial Intelligence and Intellectual Property,AI inventions present the current patent syste...,World Intellectual Property Organization (WIPO),NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,None,success
2,AI Intellectual Property,Generative AI: Navigating intellectual property,Artificial intelligence challenges the traditi...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,None,success
3,AI Intellectual Property,Intellectual Property Rights and AI-Generated ...,U.S. law does allow humans to claim copyright ...,"Medium · Adnan Masood, PhD.",NaN,https://medium.com/@adnanmasood/intellectual-p...,article,Member-only story\nIntellectual Property Right...,None,success
4,Copyright Generative AI,Copyright and Artificial Intelligence | U.S. C...,Copyright and Artificial Intelligence analyzes...,Copyright Office (.gov),NaN,https://www.copyright.gov/ai/,article,Copyright and Artificial Intelligence\nSince l...,None,success


In [15]:
df_clean

,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI Intellectual Property,"AI, Copyright, and the Law: The Ongoing Battle...",Artificial intelligence (AI) is rapidly reshap...,University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,None,success
1,AI Intellectual Property,Artificial Intelligence and Intellectual Property,AI inventions present the current patent syste...,World Intellectual Property Organization (WIPO),NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,None,success
2,AI Intellectual Property,Generative AI: Navigating intellectual property,Artificial intelligence challenges the traditi...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,None,success
3,AI Intellectual Property,Intellectual Property Rights and AI-Generated ...,U.S. law does allow humans to claim copyright ...,"Medium · Adnan Masood, PhD.",NaN,https://medium.com/@adnanmasood/intellectual-p...,article,Member-only story\nIntellectual Property Right...,None,success
4,Copyright Generative AI,Copyright and Artificial Intelligence | U.S. C...,Copyright and Artificial Intelligence analyzes...,Copyright Office (.gov),NaN,https://www.copyright.gov/ai/,article,Copyright and Artificial Intelligence\nSince l...,None,success
5,Copyright Generative AI,Copyright and Generative AI,The legal framework governing copyright faces ...,The Regulatory Review,"Jun 7, 2025",https://www.theregreview.org/2025/06/07/semina...,article,Scholars discuss emerging problems at the inte...,None,success
6,Copyright Generative AI,AI Tools and Resources: Copyright and Generati...,Generative AI tools can be used to infringe on...,University of South Florida,"Aug 21, 2026",https://guides.lib.usf.edu/AI/copyright,article,Copyright is a form of intellectual property p...,None,success
7,Copyright Generative AI,The Law and Economics of Generative AI and Cop...,Generative AI challenges copyright law at both...,National Institutes of Health (NIH) | (.gov),NaN,https://pmc.ncbi.nlm.nih.gov/articles/PMC12658...,article,Checking your browser before accessing pmc.ncb...,None,success
8,Copyright Generative AI,Copyright and Intellectual Property - Generati...,This guide features links and information abou...,The University of Virginia,"Aug 12, 2026",https://guides.lib.virginia.edu/genai/copyright,article,Connect with our knowledgeable library staff a...,None,success
9,Copyright Generative AI,Is Copyright Law the Wrong Weapon Against AI?,This year could be a critical one for court de...,Harvard Magazine,"Mar 31, 2026",https://www.harvardmagazine.com/five-questions...,article,This year could be a critical one for court de...,None,success


In [16]:
df_clean.to_csv("ai_copyright_dataset.csv", index=False)